In [ ]:
# ── Google Colab setup ────────────────────────────────────────────────────────
# Copy and run this cell first. It installs the required packages, downloads the data,
# and restarts the runtime automatically. After the restart, run all remaining
# cells normally. On a local Jupyter server this cell does nothing.
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "2026_Pipette2Code"
    NB_DIR = f"{REPO}/02_Trajectories_ZHe"

    # ── 1. Clone the course repository if not already present ─────────────────
    if not os.path.exists(REPO):
        os.system(f"git clone -q https://github.com/GIMM-BioCode/{REPO}.git")

    # ── 2. Change working directory so relative paths (data/*.h5ad) work ──────
    if os.getcwd() != f"/content/{NB_DIR}":
        os.chdir(f"/content/{NB_DIR}")
    print(f"Working directory: {os.getcwd()}")

    # ── 3. Download data files if not already present ─────────────────────────
    os.makedirs("data", exist_ok=True)
    _data_files = {
        "data/DS1.h5ad":     "https://polybox.ethz.ch/index.php/s/GCZzdxkMiTp5ZQH/download",
        "data/DS1_raw.h5ad": "https://polybox.ethz.ch/index.php/s/GeXSaepAk9fjQqk/download",
    }
    for _path, _url in _data_files.items():
        if not os.path.exists(_path):
            print(f"Downloading {_path} …")
            os.system(f"wget -q -O '{_path}' '{_url}'")
        else:
            print(f"{_path} already present.")

    # ── 4. Install packages (only when not yet at the pinned versions) ────────
    # After the restart the versions will be correct, so this block is skipped
    # and execution continues normally — no infinite restart loop.
    _needs_install = False
    try:
        import scanpy as _sc
        assert _sc.__version__ == "1.11.5"
        import scvelo as _scv
        assert _scv.__version__ == "0.3.4"
    except (ImportError, AssertionError):
        _needs_install = True

    if _needs_install:
        print("Installing required packages — the runtime will restart automatically …")
        os.system(
            f"pip install -q "+
            f"-r requirements-colab.txt "+
            "git+https://github.com/zhisonghe/pyurd.git"
        )
    else:
        print("Packages already installed — no restart needed.")

# Load the data and check information

## Load the AnnData object

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context

In [ ]:
adata = sc.read_h5ad('data/DS1.h5ad')

## Check the basic information of the data

In [ ]:
print('Shape (cells x genes):', adata.shape)
print('\nObs (cell metadata) columns:', list(adata.obs.columns))
print('\nAvailable embeddings:', list(adata.obsm.keys()))
print('\nAvailable layers:', list(adata.layers.keys()))
display(adata.obs.head())

## Rerun the preprocessing of the data
This section is to rerun data preprocessing, including normalization, HVG identification, PCA, clustering and cell type annotation, in case you want to get more familiar with how the data was processed and analyzed before doing the trajectory analysis.

***Note***\
In AnnData, it is required that `.X` and all matrices in `.layers` should share exactly the same dimensionalities. Therefore, in the `DS1.h5ad` file, only the shared features of the total counts, spliced counts and unspliced counts matrices are kept. Meanwhile, the standard analytical pipeline takes the total counts matrix to start with. Therefore, we will load the other file, `DS1_raw.h5ad`. That file includes only the raw count matrix, plus the same `.obs` data frame as the `DS1.h5ad`

In [ ]:
adata_full = sc.read_h5ad('data/DS1_raw.h5ad')
adata_full.layers['counts'] = adata_full.X.copy()
sc.pp.normalize_total(adata_full, target_sum=1e4)
sc.pp.log1p(adata_full)
sc.pp.highly_variable_genes(adata_full, n_top_genes=5000, flavor='seurat_v3', layer='counts')
sc.pp.pca(adata_full, n_comps=30, mask_var="highly_variable")
sc.pp.neighbors(adata_full)
sc.tl.umap(adata_full)
sc.tl.leiden(adata_full, resolution=1, flavor='igraph', n_iterations = 2)

In [ ]:
with rc_context({'figure.figsize': (3, 3)}):
    sc.pl.umap(adata_full, color=['SOX2', 'DCX', 'FOXG1', 'EMX1', 'DLX2', 'MKI67'],
               ncols=3, frameon=False, show=True)
    sc.pl.umap(adata_full, color=['leiden','region','celltype'],
               ncols=3, frameon=False, show=True, wspace = 0.7)

***Note***\
Here are brief information of the genes being plot above:\
SOX2: neural progenitor cell marker; DCX: neuron marker; FOXG1: telencephalic NPC-neuron marker; EMX1: dorsal telencephalic (cortical) NPC-neuron marker; DLX2: ventral telencephalic (subcortical) NPC-neuron marker; MKI67: G2M phase marker

Next, we copy the processed results to the main AnnData object, just to keep the following procedure more consistent.

***Note***\
Cells with unknown cell type annotation are not included in the main AnnData. To simplify the pipeline, here we directly subset into only cells included in the main AnnData without going through the iterative analysis to cluster and annotate the cells.

In [ ]:
adata.obsm = adata_full[adata.obs_names, :].obsm.copy()
adata.obsp = adata_full[adata.obs_names, :].obsp.copy()
adata.uns = adata_full[adata.obs_names, :].uns.copy()

# Diffusion map and diffusion pseudotime

Diffusion map can be seen as the procedure to denoise the neighbor graph. Therefore, a neighbor graph needs to be established first. With that, we will run diffusion map to calculate the first 20 diffusion components.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=50, n_pcs=30, use_rep='X_pca')
sc.tl.diffmap(adata, n_comps=20)

In scanpy, the diffusion pseudotime (`scanpy.tl.dpt`) function requires the root cell to be manually labeled. To use the same root guessing procedure as implemented in `destiny` in R, we re-implement it with the following code:

In [ ]:
import random

def random_root(adata, seed=None, neighbors_key=None, idx_subset=None):
    if seed is not None:
        random.seed(seed)
    iroot_bak = None
    if 'iroot' in adata.uns.keys():
        iroot_bak = adata.uns['iroot'].copy()
    dpt_bak = None
    if 'dpt_pseudotime' in adata.obs.columns:
        dpt_bak = adata.obs['dpt_pseudotime'].copy()

    idx = np.random.choice(list(range(adata.shape[0])))
    adata.uns['iroot'] = idx
    sc.tl.dpt(adata, neighbors_key=neighbors_key)
    dpt = adata.obs['dpt_pseudotime']
    if idx_subset is not None:
        dpt = dpt.iloc[idx_subset]
    idx_max_dpt = np.argmax(dpt)
    if idx_subset is not None:
        idx_max_dpt = idx_subset[idx_max_dpt]

    del adata.uns['iroot']
    del adata.obs['dpt_pseudotime']
    if iroot_bak is not None:
        adata.uns['iroot'] = iroot_bak.copy()
    if dpt_bak is not None:
        adata.obs['dpt_pseudotime'] = dpt_bak.copy()

    return idx_max_dpt

Now, we can restrict the root cell candidates to progenitor cells (NPC subtypes) and then perform diffusion pseudotime. The most frequently selected root across 1000 random trials is used as the final root.

In [ ]:
idx_subset = np.where(np.isin(adata.obs['celltype'], ['Dorsal telen. NPC',
                                                      'G2M dorsal telen. NPC',
                                                      'Dien. and midbrain NPC',
                                                      'G2M Dien. and midbrain NPC']))[0]
idxs_rand_root = np.apply_along_axis(lambda x: random_root(adata, idx_subset=idx_subset),
                                     1, np.array(range(1000))[:, None])
adata.uns['iroot'] = np.argmax(np.bincount(idxs_rand_root))
sc.tl.dpt(adata, n_dcs=20)

We can assign the rank of the estimated diffusion pseudotime to the AnnData object for visualization. Using the rank instead of the raw pseudotime often gives a more uniform distribution across the UMAP.

In [ ]:
from scipy.stats import rankdata

adata.obs['dpt_pseudotime_rank'] = rankdata(adata.obs['dpt_pseudotime'])

with rc_context({'figure.figsize': (3, 3)}):
    sc.pl.umap(adata, color=['dpt_pseudotime_rank', 'NES', 'MKI67', 'NHLH1', 'DCX'],
               color_map='RdBu_r', ncols=5, frameon=False)

# Coarse-grain trajectory analysis with PAGA

The implementation of PAGA, the cluster connectivity analysis which can be seen as a coarse-grain trajectory analysis, is included in `scanpy`. The analysis requires a cell-level neighborhood graph. Here, we take the diffusion map as it can be seen as a denoised neighborhood graph compared to the PCA-based one.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=50, n_pcs=20, use_rep='X_diffmap')
adata.obs['celltype'] = adata.obs['celltype'].cat.remove_unused_categories()
sc.tl.paga(adata, groups='celltype')

In [ ]:
sc.pl.paga(adata)

We can also print out the cluster pairs with their inter-cluster connectivity over a certain threshold (here, 0.1)

In [ ]:
from scipy import sparse

connected = adata.uns['paga']['connectivities'] > 0.1
connected = (connected + connected.T) > 0
idx_row, idx_col, dat = sparse.find(connected)
idx = (idx_row >= idx_col)
connected_celltypes = pd.DataFrame({'node1': adata.obs['celltype'].cat.categories[idx_row[idx]],
                                    'node2': adata.obs['celltype'].cat.categories[idx_col[idx]]})
connected_celltypes

# scVelo analysis

Next, we run scVelo for the RNA velocity analysis. It expects the input AnnData object to contain two layers: `spliced` for the spliced transcript count matrix, and `unspliced` for the unspliced transcript count matrix.

The first steps of scVelo are: 1) basic preprocessing (gene selection and normalization), and 2) computing the first- and second-order moments (means and uncentered variances) for velocity estimation

In [ ]:
import scvelo as scv

adata.raw = adata
scv.pp.filter_and_normalize(
    adata,
    min_shared_counts=20,
    min_shared_cells=20
)
sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=50)
scv.pp.moments(adata, n_neighbors=50)

Next, we perform the velocity analysis using the `stochastic` mode.

In [ ]:
scv.tl.velocity(adata, mode='stochastic')
scv.tl.velocity_graph(adata)

*P.S.* Maybe you would want to try the other two modes as well? They are `deterministic` and `dynamical`. If you want to use the `dynamical` mode, you should first run `scv.tl.recover_dynamics(adata)`, which is slower.

Now, we can visualize the estimated velocity result

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
scv.pl.velocity_embedding_stream(adata, ax=ax[0], basis='umap', color='celltype', frameon=False, show=False)
scv.pl.velocity_embedding_grid(adata, ax=ax[1], basis='umap', color='celltype', frameon=False, arrow_size=2, arrow_length=2, show=False)
scv.pl.velocity_embedding(adata, ax=ax[2], basis='umap', color='celltype', frameon=False, arrow_size=2, arrow_length=2, show=False)
plt.show()

Based on the velocity transition matrix, one can also estimate a pseudotime

In [ ]:
scv.tl.velocity_pseudotime(adata)

In [ ]:
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color=['dpt_pseudotime_rank', 'velocity_pseudotime'],
               cmap='RdBu_r', frameon=False, ncols=2)

# CytoTRACE with CellRank 2 implementation
Next, we apply CytoTRACE (with its implementation in CellRank2) to the data to measure the cell potency.

In [ ]:
from cellrank.kernels import CytoTRACEKernel

ctk = CytoTRACEKernel(adata).compute_cytotrace()

In [ ]:
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color=['dpt_pseudotime_rank', 'velocity_pseudotime','ct_pseudotime'],
               cmap='RdBu_r', frameon=False, ncols=3)

# Fate probability estimation with CellRank 2

Last but not the least, we can run CellRank by incorporating different information: transcriptomic similarity, pseudotime, velocity, and CytoTRACE

In [ ]:
import cellrank as cr

pk = cr.kernels.PseudotimeKernel(adata, time_key='dpt_pseudotime').compute_transition_matrix()
ck = cr.kernels.ConnectivityKernel(adata).compute_transition_matrix()
vk = cr.kernels.VelocityKernel(adata).compute_transition_matrix()
ctk = cr.kernels.CytoTRACEKernel(adata).compute_cytotrace().compute_transition_matrix()
combined_kernel = 0.4 * vk + 0.3 * pk + 0.2 * ck + 0.1 * ctk

*P.S.* Here, the combined kernel is 50% velocity + 30% pseudotime + 20% similarity. Maybe you can try also other combinations, or use only one source of information.

CellRank has the functionality to infer terminal states. Let's give it a try and see what population it thinks to be terminal states

In [ ]:
g = cr.estimators.GPCCA(combined_kernel)
g.fit(n_states=15, cluster_key='celltype')
g.predict_terminal_states(method='top_n', n_states=10)
g.plot_macrostates(which='terminal')

This functionality is very useful when you are working with an unfamiliar system, or you are interested in characterizing unknown cell states. The estimated terminal states are not necessarily correct, of course. When you are not convinced by the estimation, you can manually set the terminal states. Here, we set them to the neurons with the highest diffusion pseudotime

In [ ]:
g = cr.estimators.GPCCA(combined_kernel)
neuron_types = [x for x in adata.obs['celltype'].cat.categories if x.endswith('neuron') and 'early' not in x]
terminal_states = [adata[adata.obs['celltype'] == x, :].obs['dpt_pseudotime'].sort_values(ascending=False)[:30].index
                   for x in neuron_types]
terminal_states = dict(zip(neuron_types, terminal_states))
g.set_terminal_states(terminal_states)
g.plot_macrostates(which='terminal')

Now it's time to compute the fate probabilities

In [ ]:
g.compute_fate_probabilities()

In [ ]:
with rc_context({'figure.figsize': (4, 4)}):
    g.plot_fate_probabilities(legend_loc='right', basis='X_umap', same_plot=False)

We can extract the estimated probabilities into a data frame, and concatenate it with the metadata

In [ ]:
prob = pd.DataFrame(g.fate_probabilities).set_index(g.terminal_states.index).set_axis(g.terminal_states.cat.categories, axis=1)
prob.head()

In [ ]:
adata.obs = pd.concat([adata.obs, prob.set_axis(['CR_' + x for x in prob.columns], axis=1)], axis=1)

# Save the AnnData

CellRank2 generates `Lineage` objects to store the inferred lineage information and stored at `.obsm`. However, the corresponding writing method for `Lineage` objects into a .h5ad file is no longer supported in the newer version of `anndata` (>0.11). Therefore, if the saving step failed because of that, one should either install an older version of `anndata`, or convert the lineage matrices to plain numpy arrays while saving the additional names and colors attributes to `.uns`.

In [ ]:
for obsm in adata.obsm.keys():
    if type(adata.obsm[obsm]).__name__ in ['Lineage','LineageView']:
        adata.uns[str(obsm)+'_lineage'] = {
            'colors': adata.obsm[obsm].colors,
            'names': adata.obsm[obsm].names
        }
        adata.obsm[obsm] = np.array(adata.obsm[obsm])

In [ ]:
adata.write_h5ad('data/DS1_processed.h5ad')

# pyURD: Probabilistic Trajectory Inference

[pyURD](https://github.com/zhisonghe/pyurd) is a Python re-implementation of the [URD](https://github.com/farrellja/URD) trajectory-inference algorithm. It uses **flood pseudotime** propagation and **pseudotime-biased random walks** to reconstruct branching lineage trees from single-cell RNA-seq data.

> **Note:** pyURD is under active development and has no stable release yet.

This section continues directly from the earlier analyses, loading the already-processed `DS1_processed.h5ad`. Steps:
1. Flood pseudotime from root (NPC) cells
2. Pseudotime-biased transition matrix
3. Random walks from neuronal tip populations
4. Trajectory tree construction and visualisation

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# pyURD is not on PyPI — install from GitHub if needed:
#   pip install git+https://github.com/zhisonghe/pyurd.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import pyurd

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(5, 4))
print(f"scanpy {sc.__version__}  |  pyurd functions: {len(pyurd.__all__)}")

# Load the processed AnnData produced by the earlier sections
adata_urd = sc.read_h5ad("data/DS1_processed.h5ad")

print(adata_urd)
print("\nCell types:")
print(adata_urd.obs["celltype"].value_counts().to_string())

## Root and tip populations

**Root cells** are the neural progenitor cells (NPCs) — both cycling (G2M) and non-cycling.  
**Tip populations** are the five terminal neuronal lineages that define the endpoints of the branching tree.

In [ ]:
# Root cells: all NPC populations
ROOT_TYPES = [
    "Dorsal telen. NPC",
    "G2M dorsal telen. NPC",
    "Dien. and midbrain NPC",
    "G2M Dien. and midbrain NPC",
]
root_cells = adata_urd.obs_names[adata_urd.obs["celltype"].isin(ROOT_TYPES)].tolist()
print(f"Root cells: {len(root_cells)}")

# Tip populations: five terminal neuronal lineages
TIP_MAP = {
    "1": "Dorsal telen. neuron",
    "2": "Ventral telen. neuron",
    "3": "Dien. and midbrain excitatory neuron",
    "4": "Dien. and midbrain inhibitory neuron",
    "5": "Midbrain-hindbrain boundary neuron",
}

adata_urd.obs["tip"] = np.nan
for tip_id, ct in TIP_MAP.items():
    mask = adata_urd.obs["celltype"] == ct
    adata_urd.obs.loc[mask, "tip"] = tip_id

print("Tip cell counts:")
print(adata_urd.obs["tip"].value_counts().sort_index())

sc.pl.umap(adata_urd, color="tip", title="Tip populations")

## Flood pseudotime

pyURD propagates pseudotime by running *n* probabilistic BFS simulations forward from the root cells. `pyurd.flood_pseudotime_process` aggregates the simulations into a single normalised pseudotime value per cell.

The transition matrix used here is the diffusion-map-denoised connectivity graph generated with `pyurd.prepare_connectivities`. Alternatively, the raw k-NN connectivity graph (`connectivities`) can also be used.

In [ ]:
# Run 50 flood simulations from root cells using the k-NN connectivity graph
# (n_jobs=-1 uses all CPU cores via joblib)
transition_key = pyurd.prepare_connectivities(adata_urd, n_neighbors=50, n_dcs=30)
floods = pyurd.flood_pseudotime(
    adata_urd,
    root_cells=root_cells,
    n=50,
    minimum_cells_flooded=2,
    transition_key=transition_key,
    verbose=False,
    n_jobs=-1,
)
print(f"Floods shape: {floods.shape}  (cells × simulations)")
floods.head()

In [ ]:
# Aggregate simulations into a single pseudotime value per cell
pyurd.flood_pseudotime_process(
    adata_urd,
    floods=floods,
    floods_name="pseudotime",
    max_frac_na=0.4,
    stability_div=10,
)

print("Pseudotime stored in adata_urd.obs['pseudotime']")
print(adata_urd.obs["pseudotime"].describe())

sc.pl.umap(
    adata_urd,
    color=["pseudotime",'dpt_pseudotime_rank', 'velocity_pseudotime','ct_pseudotime'],
    color_map="RdBu_r", ncols=4, frameon=False, title=["Flood pseudotime", "DPT pseudotime rank", "Velocity pseudotime", "CytoTRACE pseudotime"]
)

In [ ]:
# Correlations between pseudotime measures
corr_df = adata_urd.obs[["pseudotime", "dpt_pseudotime_rank", "velocity_pseudotime","ct_pseudotime"]].corr(method='spearman')
print("Correlation matrix:")
print(corr_df.to_string())

In [ ]:
# Pseudotime stability: shows convergence as more simulations are added
stability_df = adata_urd.uns["urd"]["pseudotime_stability"]["pseudotime"]["pseudotime"]

windows = sorted(stability_df.columns)
mean_diffs = [
    (stability_df[windows[i]] - stability_df[windows[i - 1]]).abs().mean()
    for i in range(1, len(windows))
]

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(windows[1:], mean_diffs, marker="o", ms=4)
ax.set_xlabel("Number of simulations")
ax.set_ylabel("Mean pseudotime difference")
ax.set_title("Pseudotime stability")
ax.axhline(0, color="grey", lw=0.8, ls="--")
plt.tight_layout()
plt.show()

## Pseudotime-biased transition matrix

To simulate random walks that move *away* from the root (toward terminal fates), we bias the transition matrix so that steps to younger (lower pseudotime) cells are penalised. `pyurd.pseudotime_determine_logistic` fits a logistic bias function; `pyurd.pseudotime_weight_transition_matrix` applies it to the k-NN connectivity.

In [ ]:
# Fit logistic parameters:
#   optimal_cells_forward=20  → pseudotime gap of 20 cells forward ≈ prob 1
#   max_cells_back=40         → pseudotime gap of 40 cells backward ≈ prob 0.01
logistic_params = pyurd.pseudotime_determine_logistic(
    adata_urd,
    pseudotime_key="pseudotime",
    optimal_cells_forward=20,
    max_cells_back=40,
    do_plot=True,
    verbose=True,
)
print("Logistic parameters:", logistic_params)

# Build the biased transition matrix
biased_tm = pyurd.pseudotime_weight_transition_matrix(
    adata_urd,
    pseudotime_key="pseudotime",
    logistic_params=logistic_params,
    transition_key=transition_key,
    chunk_size=500,
)
print(f"Biased TM shape: {biased_tm.shape}  dtype: {biased_tm.dtype}")

## Random walks from tip populations

Random walks start from each tip population and walk backwards through pseudotime toward the root. Visitation frequencies accumulate across walks, encoding how often each cell is visited on the path to each neuronal fate.

In [ ]:
# Register tip cells
pyurd.load_tip_cells(adata_urd, tips_key="tip")

# Simulate 10 000 random walks per tip
walks = pyurd.simulate_random_walks_from_tips(
    adata_urd,
    tip_group_key="tip",
    root_cells=root_cells,
    transition_matrix=biased_tm,
    n_per_tip=10_000,
    root_visits=3,
    max_steps=5000,
    verbose=True,
)

for tip_id, tip_walks in walks.items():
    ct = TIP_MAP[tip_id]
    print(f"  Tip {tip_id} ({ct}): {len(tip_walks)} walks")

In [ ]:
# Convert walks to per-cell visitation frequencies stored in adata_urd.obs
pyurd.process_random_walks_from_tips(adata_urd, walks_dict=walks, verbose=True)

visit_cols = [c for c in adata_urd.obs.columns if c.startswith("visitfreq_log_")]
print("Visit frequency columns:", visit_cols)

sc.pl.umap(
    adata_urd,
    color=visit_cols,
    color_map="YlOrRd",
    ncols=3,
    vmin=0,
)

## Trajectory tree

`pyurd.build_tree` agglomeratively joins the five tip trajectories by finding the pseudotime at which their visitation patterns diverge. The result — segment assignments, join points, and the tree topology — is stored in `adata_urd.uns['urd']`.

In [ ]:
pyurd.build_tree(
    adata_urd,
    pseudotime_key="pseudotime",
    tips_use=list(TIP_MAP.keys()),   # ["1", "2", "3", "4", "5"]
    divergence_method="preference",  # uses Hartigan dip test
    weighted_fusion=True,
    cells_per_pseudotime_bin=80,
    bins_per_pseudotime_window=5,
    minimum_visits=10,
    visit_threshold=0.7,
    p_thresh=0.01,
    dendro_node_size=100,
    verbose=True,
)

print("\nSegments in the tree:", adata_urd.uns["urd"]["segments"])
print("\nSegment joins:")
print(adata_urd.uns["urd"]["segment_joins"])

In [ ]:
# Assign human-readable names to terminal segments
pyurd.name_segments(
    adata_urd,
    segments=["1", "2", "3", "4", "5"],
    segment_names=[
        "dTelen Neuron",
        "vTelen Neuron",
        "Dien/MB Exc. Neuron",
        "Dien/MB Inh. Neuron",
        "MHB Neuron",
    ],
    short_names=["dTN", "vTN", "DiMB-E", "DiMB-I", "MHB"],
)

print("Named segments:", adata_urd.uns["urd"]["segment_names"])

In [ ]:
# Plot tree coloured by cell type, pseudotime, and region
fig, axes = plt.subplots(1, 3, figsize=(16, 8))

pyurd.plot_tree(adata_urd, label="celltype",   title="Cell type",   ax=axes[0], legend=False, cell_size=4)
pyurd.plot_tree(adata_urd, label="pseudotime", title="Pseudotime",  ax=axes[1], legend=True,  cell_size=4)
pyurd.plot_tree(adata_urd, label="region",     title="Region",      ax=axes[2], legend=True,  cell_size=4)

plt.tight_layout()
plt.show()

# Tree and UMAP coloured by segment
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

pyurd.plot_tree(adata_urd, label="segment", title="Segments (tree)", ax=axes[0], legend=True, cell_size=4)
sc.pl.umap(adata_urd, color="segment", title="Segments (UMAP)", ax=axes[1], show=False)
axes[1].set_aspect("auto")

plt.tight_layout()
plt.show()

In [ ]:
# Tree navigation helpers
urd = adata_urd.uns["urd"]

terminals = pyurd.seg_terminal(urd)
print("Terminal segments:", terminals)

root_seg = urd["segments"][-1]
print(f"\nRoot segment: {root_seg}")
print("Direct children:", pyurd.seg_children(urd, root_seg))
print("All descendants:", pyurd.seg_children_all(urd, root_seg))

# Trace path from first terminal to root
tip_seg = terminals[0]
path = [tip_seg]
while True:
    parent = pyurd.seg_parent(urd, path[-1])
    if parent is None or parent == path[-1]:
        break
    path.append(parent)
seg_names = urd.get("segment_names", {})
path_named = [seg_names.get(s, s) for s in path]
print(f"\nPath from tip '{tip_seg}' to root: {' → '.join(path_named)}")

## Save and reload

`pyurd.save_urd` / `pyurd.load_urd` serialise and restore the full AnnData including the URD tree stored in `adata.uns['urd']`, so the pipeline does not need to be re-run from scratch.

In [ ]:
pyurd.save_urd(adata_urd, "data/DS1_pyurd.h5ad")
print("Saved to data/DS1_pyurd.h5ad")

In [ ]:
adata_urd2 = pyurd.load_urd("data/DS1_pyurd.h5ad")

print(adata_urd2)
print("\nSegments:", adata_urd2.uns["urd"]["segments"])
print("Segment names:", adata_urd2.uns["urd"].get("segment_names", {}))

# Verify tree plots work on the reloaded object
fig, axes = plt.subplots(1, 3, figsize=(16, 8))

pyurd.plot_tree(adata_urd2, label="celltype",   title="Cell type",   ax=axes[0], legend=False, cell_size=4)
pyurd.plot_tree(adata_urd2, label="pseudotime", title="Pseudotime",  ax=axes[1], legend=True,  cell_size=4)
pyurd.plot_tree(adata_urd2, label="segment",    title="Segments",    ax=axes[2], legend=True,  cell_size=4)

plt.tight_layout()
plt.show()